# A single qubit with PennyLane

This tutorial follows the Qiskit examples in `single_system.ipynb`.
You will represent a qubit, predict and sample measurement outcomes, apply
custom gates, and build a circuit whose matrix you can check by hand.
Basic Python and matrix multiplication are enough to follow along.

**How to run:** open `single_system_pennylane.ipynb` in VS Code or Jupyter
and execute its cells from top to bottom. This `.ipy` file contains the same
tutorial as commented Markdown and executable Python. In IPython, run it with
`%run single_system_pennylane.ipy`.
If dependencies are missing, run
`%pip install pennylane numpy matplotlib sympy ipython` in a notebook cell,
then restart the kernel. Qiskit is not required for this tutorial.

## 1. Setup

We use PennyLane for quantum simulation, NumPy for vectors and matrices,
and Matplotlib for plots. SymPy and IPython only format the displayed math.
The formatting helpers below do not change the simulated states.

The `is_valid` helper checks a two-component vector and its squared norm.
`np.vdot` conjugates its first argument, so it correctly handles complex
amplitudes. Numerical checks use a tolerance for floating-point rounding.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pennylane as qml
import sympy as sp
from IPython.display import Latex, display
from numpy import sqrt


def is_valid(state):
    """Check that a single-qubit state has unit norm."""
    state = np.asarray(state, dtype=complex)
    return bool(
        state.shape == (2,)
        and np.all(np.isfinite(state))
        and np.isclose(np.vdot(state, state), 1.0)
    )


def scalar_latex(value):
    return sp.latex(sp.nsimplify(complex(value), [sp.sqrt(2), sp.sqrt(5)]))


def state_latex(state):
    terms = [
        rf"\left({scalar_latex(amplitude)}\right)|{index}\rangle"
        for index, amplitude in enumerate(state)
        if not np.isclose(amplitude, 0)
    ]
    return " + ".join(terms) or "0"


def matrix_latex(matrix):
    rows = [" & ".join(scalar_latex(value) for value in row) for row in matrix]
    return r"\begin{bmatrix}" + r" \\ ".join(rows) + r"\end{bmatrix}"


## 2. Represent a quantum state

A single-qubit pure state is
$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$.
Its array is `[alpha, beta]`: the entries are **amplitudes**, not probabilities.
The Born rule gives $P(0)=|\alpha|^2$ and $P(1)=|\beta|^2$.
A valid state therefore satisfies $|\alpha|^2+|\beta|^2=1$.

Predict the results for these three examples before running the cell:

| Vector | Squared norm | Valid? | Measurement probabilities |
| --- | --- | --- | --- |
| `u` | 1 | Yes | 1/2, 1/2 |
| `v` | 1 | Yes | 5/9, 4/9 |
| `w` | 5/9 | No | Must normalize first |

For `v`, $|(1+2i)/3|^2=(1^2+2^2)/9=5/9$.
The minus sign on its second amplitude disappears when taking the squared
magnitude, but phases can affect interference after later gates.


In [ ]:
# PennyLane accepts state vectors as arrays; StatePrep loads them into a circuit.
u = np.array([1 / sqrt(2), 1 / sqrt(2)], dtype=complex)
v = np.array([(1 + 2.0j) / 3, -2 / 3], dtype=complex)
w = np.array([1 / 3, 2 / 3], dtype=complex)

display(u)
display(Latex("$$" + state_latex(u) + "$$"))
print(state_latex(u))

display(is_valid(u))  # True
display(is_valid(w))  # False: deliberately not normalized
print("Probabilities for v:", np.abs(v) ** 2)
# Keep this input under a descriptive name for later experiments.
measurement_input = v.copy()


## 3. Take one measurement

A PennyLane **device** executes the circuit. Here `default.qubit` is a local
simulator and `wires=1` creates one qubit, labelled `0`.
A **QNode** connects a Python quantum function to that device.
Inside the function, `StatePrep` prepares the input and `qml.sample` requests
computational-basis outcomes. Calling the QNode executes the computation.
See the [PennyLane circuit guide](https://docs.pennylane.ai/en/stable/introduction/circuits.html).

One **shot** is one preparation and measurement. We use
[`qml.set_shots`](https://docs.pennylane.ai/en/stable/code/api/pennylane.set_shots.html)
to request a single shot. Either outcome is possible: `0` with probability
5/9, or `1` with probability 4/9.

For outcome $k$, the conditional state is
$|\psi_k\rangle=|k\rangle\langle k|\psi\rangle/\sqrt{P(k)}$.
Our helper reconstructs this state from the returned sample; it is not a
state returned by `qml.sample`. It also leaves the input array unchanged.
For outcome `0`, the result is $(1+2i)/\sqrt{5}\,|0\rangle$;
for outcome `1`, it is $-|1\rangle$. Each differs from its basis ket only
by an unobservable global phase. Keeping that phase matches the source example.


In [ ]:
device = qml.device("default.qubit", wires=1)


@qml.qnode(device)
def sample_state(initial_state):
    qml.StatePrep(initial_state, wires=[0], validate_norm=True)
    return qml.sample(wires=[0])


def measure_state(initial_state):
    """Sample once and reconstruct the normalized projected state."""
    initial_state = np.asarray(initial_state, dtype=complex)
    sample = qml.set_shots(sample_state, shots=1)(initial_state)
    index = int(np.asarray(sample).item())
    # A terminal sample does not return the collapsed state. Apply the selected
    # basis projector explicitly, retaining the phase as Qiskit's measure does.
    collapsed_state = np.zeros_like(initial_state)
    collapsed_state[index] = initial_state[index] / abs(initial_state[index])
    return str(index), collapsed_state


print("State before measurement:")
display(Latex("$$" + state_latex(v) + "$$"))
outcome, state = measure_state(v)
print(f"Measured: {outcome}\nPost-measurement state:")
display(Latex("$$" + state_latex(state) + "$$"))


## 4. Repeat the experiment

Reprepare the original `v` for every shot. Measuring a fresh copy is different
from repeatedly measuring one already-collapsed qubit, which would keep giving
the same basis outcome if no gates intervene.

`qml.counts` returns a dictionary such as `{'0': 556, '1': 444}`.
Your counts will vary. Dividing by the number of shots gives relative
frequencies, which should be near 5/9 and 4/9. More shots usually improve
agreement; they do not guarantee a closer result on every run.

**Try it:** change `shots` to 10, then 10000, and compare the bars with the
orange crosses showing the exact probabilities. This cell uses
`measurement_input` so it is safe to rerun after later cells redefine `v`.


In [ ]:
@qml.qnode(device)
def sample_counts(initial_state):
    qml.StatePrep(initial_state, wires=[0], validate_norm=True)
    return qml.counts(wires=[0], all_outcomes=True)


shots = 1000
statistics = qml.set_shots(sample_counts, shots=shots)(measurement_input)
print("Counts:", statistics)
labels = sorted(statistics)
probabilities = [statistics[label] / shots for label in labels]
fig, ax = plt.subplots()
bars = ax.bar(labels, probabilities, color="blue", label="Sampled frequency")
ax.scatter(labels, np.abs(measurement_input) ** 2,
           color="darkorange", marker="x", s=100, label="Exact probability")
ax.bar_label(bars, fmt="%.3f")
ax.set(title="Measurement statistics", xlabel="Outcome", ylabel="Probability")
ax.set_ylim(0, 1)
ax.legend()
plt.show()


## 5. Apply custom unitary matrices

A gate changes a state by matrix multiplication: $|\psi'\rangle=U|\psi\rangle$.
A unitary matrix satisfies $U^\dagger U=I$, preserving the norm.
Here $\dagger$ means conjugate transpose, written `U.conj().T` in NumPy.

| Gate | Action |
| --- | --- |
| H | Maps basis states to equal superpositions; applying H twice gives I |
| T | Multiplies the amplitude of 1 by $e^{i\pi/4}$ |
| S | Multiplies the amplitude of 1 by $i$ |
| Y | Sends $|0\rangle$ to $i|1\rangle$ and $|1\rangle$ to $-i|0\rangle$ |

`qml.QubitUnitary` applies a supplied matrix. `qml.state()` returns the
simulator's complex state vector. This is an analytic simulator result, not
something one shot on hardware reveals. We leave this QNode in analytic mode
with no finite shot count. See the
[`qml.state` reference](https://docs.pennylane.ai/en/stable/code/api/pennylane.state.html).

Following the source, apply H, T, H, S to $|0\rangle$, leaving Y commented
out. The expected amplitudes are approximately
`[0.853553 + 0.353553j, 0.353553 + 0.146447j]`.
The second H converts the phase introduced by T into unequal probabilities.


In [ ]:
Y = np.array([[0, -1.0j], [1.0j, 0]], dtype=complex)
H = np.array([[1 / sqrt(2), 1 / sqrt(2)],
              [1 / sqrt(2), -1 / sqrt(2)]], dtype=complex)
S = np.array([[1, 0], [0, 1.0j]], dtype=complex)
T = np.array([[1, 0], [0, (1 + 1.0j) / sqrt(2)]], dtype=complex)

display(Latex("$$" + matrix_latex(T) + "$$"))


@qml.qnode(device)
def evolve(initial_state, unitary):
    qml.StatePrep(initial_state, wires=[0], validate_norm=True)
    qml.QubitUnitary(unitary, wires=[0])
    return qml.state()


v = np.array([1, 0], dtype=complex)
v = evolve(v, H)
v = evolve(v, T)
v = evolve(v, H)
v = evolve(v, S)
# v = evolve(v, Y)

display(Latex("$$" + state_latex(v) + "$$"))


## 6. Build the same operations as a circuit

Named gates express the operation sequence directly. The function `circuit`
below is a reusable block of gates: it will be called inside a QNode in the
next section. We can also draw it and extract its matrix.

This section includes a final Y, just as the original notebook does.
It therefore differs from the previous section by that gate.

Gates execute in code order, H then T then H then S then Y. With column
vectors, the rightmost matrix acts first, so the total operator is
$U=YSHTH$. Compare `qml.matrix` with `Y @ S @ H @ T @ H` below.
The empty parentheses after `qml.matrix(...)` invoke the transformed function;
`wire_order=[0]` explicitly sets the qubit order for its matrix.


In [ ]:
def circuit():
    qml.Hadamard(wires=0)
    qml.T(wires=0)
    qml.Hadamard(wires=0)
    qml.S(wires=0)
    qml.PauliY(wires=0)


fig, ax = qml.draw_mpl(circuit)()
plt.show()

print("Operation of the circuit:")
operation = qml.matrix(circuit, wire_order=[0])()
display(Latex("$$" + matrix_latex(operation) + "$$"))
print("Matrix matches Y @ S @ H @ T @ H:",
      np.allclose(operation, Y @ S @ H @ T @ H))


## 7. Execute the circuit on an initial state

The QNode first prepares `ket0`, then runs the gate block and returns the state.
Compare that output with ordinary matrix multiplication, `operation @ ket0`.
Both descriptions implement the same transformation.

Expected amplitudes are approximately
`[0.146447 - 0.353553j, -0.353553 + 0.853553j]`.
Their squared magnitudes are about `[0.146447, 0.853553]`.
The final Y exchanges the probabilities from the H-T-H-S example and changes
phases. State amplitudes remain complex even though probabilities are real.


In [ ]:
@qml.qnode(device)
def run_circuit(initial_state):
    qml.StatePrep(initial_state, wires=[0], validate_norm=True)
    circuit()
    return qml.state()


ket0 = np.array([1, 0], dtype=complex)
v = run_circuit(ket0)
display(Latex("$$" + state_latex(v) + "$$"))
print("Final amplitudes:", v)
print("Final probabilities:", np.abs(v) ** 2)
print("State matches matrix multiplication:", np.allclose(v, operation @ ket0))
print("Final state is normalized:", is_valid(v))


## 8. Exercises

Try each prediction before executing the corresponding code in a new cell.

1. Normalize `w` with `w / np.linalg.norm(w)`. What are its probabilities?
2. Apply H twice to `ket0` using `evolve`. What state returns?
3. Compare `np.abs(evolve(u, T)) ** 2` with `np.abs(u) ** 2`.
   Then compare H applied to each state. When does the phase become visible?
4. Run `run_circuit(np.array([0, 1], dtype=complex))`.
   Which column of `operation` should you get?
5. Sample the final circuit state with
   `qml.set_shots(sample_counts, shots=1000)(v)`.
   Which outcome should be more common?

### Check your reasoning

1. The normalized state is `[1/sqrt(5), 2/sqrt(5)]`, with probabilities 1/5, 4/5.
2. $H^2=I$, so you recover $|0\rangle$.
3. T alone keeps the probabilities at 1/2, 1/2. After H, the original `u`
   gives 1, 0; the T-transformed state gives approximately 0.853553, 0.146447.
4. You get column index 1 (the second column), because the input is $|1\rangle$.
5. Outcome `1`, with probability approximately 0.853553.

## Qiskit-to-PennyLane reference

| Original example | Equivalent used here |
| --- | --- |
| `Statevector([...])` | Complex NumPy array and `qml.StatePrep` |
| `state.is_valid()` | `is_valid(state)` helper |
| `state.draw('latex')` | `state_latex` and IPython `Latex` helpers |
| `state.measure()` | One-shot `qml.sample` and explicit conditional projection |
| `state.sample_counts(shots=1000)` | `qml.counts` with `qml.set_shots` |
| `Operator(matrix)` | NumPy matrix applied by `qml.QubitUnitary` |
| `state.evolve(operator)` | `evolve` QNode returning `qml.state()` |
| `QuantumCircuit` and gate methods | Gate function inside a QNode |
| `circuit.draw('mpl')` | `qml.draw_mpl(circuit)()` |
| `Operator.from_circuit(circuit)` | `qml.matrix(circuit, wire_order=[0])()` |
